# S2·JUE — Evaluar recuperación sobre un libro entero

**MMIA 6013 · Semana 2 · Jueves** — ensayo general del Taller 2.

Hasta hoy el corpus fueron tres párrafos inventados. Hoy el corpus es **un libro del
curso**: *Hands-On Large Language Models* (Alammar & Grootendorst, 2024), 393 páginas
con texto, y el pipeline completo corre sobre él de principio a fin.

1. Trocear el libro conservando el **folio**, que es lo que se cita
2. Las tres métricas: Hit Rate@k, Recall@k y MRR
3. Un golden set con anclas verificables, y dos preguntas sin respuesta
4. **Cuatro motores**: BM25 · densa · híbrida (RRF) · +reranking con cross-encoder
5. La tabla que compara los cuatro con números y con tiempos

El libro trae su propio capítulo 8 sobre búsqueda semántica, reranking y métricas de
recuperación: hoy se evalúa un buscador **sobre el texto que explica cómo evaluarlo**.

Requisitos: `numpy`, `sentence-transformers` (ya instalado). Sin red ni claves.

## 🧑‍🏫 Notas docentes

- **El corpus ya está extraído**: `fuentes/libros/new/Hands-On_Large_Language_Models.md`,
  732 KB de markdown con un ancla `<!-- p. F · pdf N -->` por página. No se parsea el PDF
  en clase: son dos minutos y medio de docling que nadie quiere esperar con el aula
  delante. El ancla es lo que permite que una respuesta se cite por **folio**.
- **Números medidos el 2026-09-23 en un Mac**, para dimensionar la clase: trocear 0,3 s ·
  BM25 indexa en 0,03 s · el modelo denso codifica los 954 chunks en **2 s** · el
  cross-encoder reordena 30 candidatos en 0,06 s. La primera vez, bajar los dos modelos
  son ~20 s. Todo el cuaderno corre en menos de un minuto.
- **El resultado no es monótono y NO hay que esconderlo.** En la medición de referencia,
  híbrido (1,00 hit@5) gana a denso (0,92) y a BM25 (0,83), pero **el reranking baja a
  0,92**. Es el momento de la clase: un reranker entrenado en MS MARCO —consultas cortas
  de buscador— sobre prosa de libro técnico no siempre mejora. La pregunta correcta no es
  «¿el reranking sirve?» sino «¿sobre qué distribución se entrenó, y se parece a la mía?».
- Si el aula va lenta, bajar `PAGINAS_MAX` a 150: el pipeline es idéntico y las métricas
  siguen discriminando.
- Lo que este cuaderno **hereda del miércoles** y conviene decir en voz alta: la raíz del
  repositorio se busca hacia arriba (no con rutas relativas fijas), y cada motor imprime
  qué recuperó antes de imprimir su promedio. Un promedio sin ejemplos no se puede
  depurar.

In [1]:
import math
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np


def _raiz_repo(inicio: Path) -> Path:
    """Sube directorios hasta la raíz del repositorio del curso.

    La ruta relativa fija falla: desde `curso/actividades/semana-N/` dos niveles
    arriba es `curso/`, no la raíz. Buscar hacia arriba hace además que el
    notebook corra desde donde se lo abra.
    """
    for base in [inicio, *inicio.parents]:
        if (base / "curso/actividades").is_dir():
            return base
    raise FileNotFoundError(
        "No encuentro `curso/actividades/`: abre el notebook dentro del "
        "repositorio del curso.")


RAIZ = _raiz_repo(Path.cwd())
LIBRO = RAIZ / "fuentes/libros/new/Hands-On_Large_Language_Models.md"
PAGINAS_MAX = None      # None = el libro entero; 150 para un aula con prisa
TAM, SOLAPA = 900, 200  # caracteres por chunk y solapamiento
K = 5                   # el corte de todas las métricas

print(f"raíz del curso : {RAIZ}")
print(f"corpus         : {LIBRO.relative_to(RAIZ)} ({LIBRO.stat().st_size // 1024} KB)")

raíz del curso : /Users/driofrio/Agents/GenAIV3
corpus         : fuentes/libros/new/Hands-On_Large_Language_Models.md (714 KB)


## Parte 0 — El corpus: un libro, troceado sin perder el folio

El markdown trae un ancla por página. Se trocea **dentro de cada página**, no a través de
ellas, y cada chunk se queda con su folio. Eso cuesta algún chunk corto al final de cada
página y compra una cosa que vale más: **toda respuesta se puede citar**, y el golden set
se puede anotar con un número que alguien puede ir a comprobar al PDF.

In [2]:
ANCLA = re.compile(r"<!--\s*p\.\s*(\d+)\s*·\s*pdf\s*(\d+)\s*-->")


def cargar_paginas(ruta: Path, maximo=None):
    """[{folio, pdf, texto}] — una entrada por página con texto suficiente."""
    partes = ANCLA.split(ruta.read_text(encoding="utf-8"))
    paginas = []
    # El split deja: [preámbulo, folio, pdf, cuerpo, folio, pdf, cuerpo, ...]
    for i in range(1, len(partes), 3):
        cuerpo = re.sub(r"<!--.*?-->", " ", partes[i + 2], flags=re.S)  # otros comentarios
        cuerpo = re.sub(r"\s+", " ", cuerpo).strip()
        if len(cuerpo) > 200:          # portadillas y páginas de figuras quedan fuera
            paginas.append({"folio": int(partes[i]), "pdf": int(partes[i + 1]),
                            "texto": cuerpo})
    return paginas[:maximo] if maximo else paginas


def ventanas(texto: str, tam=TAM, solapa=SOLAPA):
    """Ventana fija con solapamiento, DENTRO de una página."""
    trozos, paso = [], tam - solapa
    for ini in range(0, len(texto), paso):
        t = texto[ini:ini + tam].strip()
        if len(t) > 100:
            trozos.append(t)
        if ini + tam >= len(texto):
            break
    return trozos


t0 = time.perf_counter()
PAGINAS = cargar_paginas(LIBRO, PAGINAS_MAX)
CHUNKS = [{"texto": t, "folio": p["folio"], "pdf": p["pdf"]}
          for p in PAGINAS for t in ventanas(p["texto"])]
TEXTOS = [c["texto"] for c in CHUNKS]
FOLIOS = [c["folio"] for c in CHUNKS]

print(f"{len(PAGINAS)} páginas · folios {PAGINAS[0]['folio']}–{PAGINAS[-1]['folio']} · "
      f"{time.perf_counter() - t0:.2f} s")
print(f"{len(CHUNKS)} chunks · {np.mean([len(t) for t in TEXTOS]):.0f} caracteres de media")
print(f"\nun chunk cualquiera (folio {CHUNKS[400]['folio']}, pdf {CHUNKS[400]['pdf']}):")
print(f"  {TEXTOS[400][:200]}…")

393 páginas · folios 37–405 · 0.02 s
954 chunks · 760 caracteres de media

un chunk cualquiera (folio 175, pdf 197):
  We can continue adding or updating the elements of a prompt until we elicit the response we are looking for. We could add additional examples, describe the use case in more detail, provide additional …


## Parte 1 — Las tres métricas

### ✏️ Ejercicio 1.1
Implementa sobre una lista de rankings (documentos ordenados por score) y la
lista de relevantes de cada pregunta:
- `hit_rate_k(rankings, relevantes, k)`
- `recall_k(rankings, relevantes, k)`
- `mrr(rankings, relevantes, k)` — con el MISMO corte que las otras dos

Las tres se calculan **solo sobre las preguntas respondibles** (`rel` no vacío):
una negativa no tiene nada que recuperar, y dejarla en el denominador pone un techo
falso al Hit Rate (con 2 negativas en 14, un sistema perfecto reporta 0,86). Para las
negativas implementa aparte:
- `tasa_abstencion(scores_top1, relevantes, umbral)`: fracción de negativas cuyo mejor
  score queda **por debajo** del umbral (abstención correcta) y fracción de
  respondibles cuyo mejor score también queda por debajo (abstención indebida). Son
  dos tasas, no una.

In [3]:
# Ejercicio 1.1 — métricas de retrieval en ~15 líneas
def hit_rate_k(rankings, relevantes, k=3):
    pares = [(r, rel) for r, rel in zip(rankings, relevantes) if rel]   # solo respondibles
    return float(np.mean([len(set(r[:k]) & set(rel)) > 0 for r, rel in pares]))

def recall_k(rankings, relevantes, k=3):
    pares = [(r, rel) for r, rel in zip(rankings, relevantes) if rel]   # solo respondibles
    return float(np.mean([len(set(r[:k]) & set(rel)) / len(rel) for r, rel in pares]))

def mrr(rankings, relevantes, k=3):
    # El corte en k es el mismo de las otras dos, y el mismo del evaluador del Lab 02:
    # alli el ranking que llega ya viene truncado por retrieve(top_k=k). Sin el corte, un
    # acierto en la posicion 20 sumaba 1/20 al MRR mientras el Hit Rate@k lo contaba como
    # fallo, y entonces MRR > Hit Rate@k es posible: el plano de la sesion 09 declara ese
    # cuadrante imposible.
    rrs = []
    for r, rel in zip(rankings, relevantes):
        if not rel:            # negativa: no entra en el promedio
            continue
        rr = 0.0
        for pos, doc in enumerate(r[:k], start=1):
            if doc in rel:
                rr = 1.0 / pos
                break
        rrs.append(rr)
    return float(np.mean(rrs))

def tasa_abstencion(scores_top1, relevantes, umbral):
    """Dos tasas: correcta (negativas bajo el umbral) e indebida (respondibles bajo el umbral)."""
    neg = [s < umbral for s, rel in zip(scores_top1, relevantes) if not rel]
    pos = [s < umbral for s, rel in zip(scores_top1, relevantes) if rel]
    return {"abstencion_correcta": float(np.mean(neg)) if neg else float("nan"),
            "abstencion_indebida": float(np.mean(pos)) if pos else float("nan")}

# sanity check con un caso a mano
assert mrr([["a", "b", "c"]], [["b"]], k=3) == 0.5
# fuera del corte no suma: con k=1 el acierto en la posicion 2 vale 0, igual que el
# Hit Rate@1. Es lo que hace que MRR@k <= Hit Rate@k nunca se rompa.
assert mrr([["a", "b", "c"]], [["b"]], k=1) == 0.0
assert hit_rate_k([["a", "b", "c"]], [["c"]], k=2) == 0.0
assert recall_k([["a"], ["b"]], [["a"], []], k=1) == 1.0      # la negativa no cuenta
assert tasa_abstencion([0.9, 0.2], [["a"], []], umbral=0.5) == {"abstencion_correcta": 1.0, "abstencion_indebida": 0.0}
print("métricas OK ✓")

métricas OK ✓


## Parte 2 — El golden set, y cómo se defiende

Doce preguntas respondibles y dos negativas. Lo que hace a este golden set **defendible**
no es que las preguntas sean buenas: es que cada `rel` es una lista de **folios que
alguien puede ir a abrir en el PDF**. Se construyó al revés de como se suele: primero se
localizó dónde el libro explica cada concepto, y después se escribió la pregunta.

Las preguntas están repartidas en tres grupos a propósito, y el reparto es lo que hace
que la comparación de la Parte 7 signifique algo:

| Grupo | Qué mide | Ejemplo |
|---|---|---|
| **léxicas** | el término de la pregunta está literal en el texto | «What is byte pair encoding?» |
| **de vocabulario distinto** | el concepto está, la palabra no | «make a model use less memory…» → *quantization* |
| **negativas** | que el sistema sepa callarse | «How much does Qdrant Cloud cost?» |

Un golden set sin el segundo grupo hace ganar siempre a BM25. Uno sin el tercero no
detecta jamás un sistema que responde a todo.

In [4]:
GOLDEN = [
    # ── léxicas: la palabra de la pregunta está en el texto ────────────────────────
    {"q": "What is byte pair encoding?", "rel": [43, 49, 51, 53, 55], "grupo": "léxica"},
    {"q": "What is negative sampling in word2vec training?", "rel": [65, 66, 71], "grupo": "léxica"},
    {"q": "What is the difference between a bi-encoder and a cross-encoder?",
     "rel": [294, 295, 311, 312], "grupo": "léxica"},
    {"q": "What is dense retrieval?", "rel": [228, 229, 230], "grupo": "léxica"},
    {"q": "What is reranking and when should I use it?", "rel": [240, 241, 242, 243], "grupo": "léxica"},
    {"q": "What is a vector database used for?", "rel": [229, 230, 238, 239], "grupo": "léxica"},
    # ── vocabulario distinto: el concepto está, la palabra no ──────────────────────
    {"q": "How can I make a model use less memory by lowering the precision of its weights?",
     "rel": [365, 366, 367], "grupo": "vocabulario"},          # quantization
    {"q": "How do I adapt a large model without updating all of its parameters?",
     "rel": [359, 360, 361, 362, 363, 364, 365, 366], "grupo": "vocabulario"},   # PEFT / LoRA
    {"q": "Why do language models confidently state information that is false?",
     "rel": [177, 225, 227], "grupo": "vocabulario"},          # hallucination
    {"q": "How do I measure the quality of a search system?",
     "rel": [244, 246, 247, 248, 249], "grupo": "vocabulario"},  # MAP, nDCG
    {"q": "How should I split long documents before indexing them?",
     "rel": [235, 236, 237, 238], "grupo": "vocabulario"},     # chunking
    {"q": "What happens when keyword search misses a relevant document?",
     "rel": [233, 235], "grupo": "vocabulario"},
    # ── negativas: no están en el libro ────────────────────────────────────────────
    {"q": "How much does Qdrant Cloud cost per month?", "rel": [], "grupo": "negativa"},
    {"q": "How do I configure horizontal pod autoscaling in Kubernetes?", "rel": [], "grupo": "negativa"},
]

RESPONDIBLES = [g for g in GOLDEN if g["rel"]]
print(f"{len(GOLDEN)} preguntas · {len(RESPONDIBLES)} respondibles · "
      f"{len(GOLDEN) - len(RESPONDIBLES)} negativas")
for grupo in ("léxica", "vocabulario", "negativa"):
    print(f"   {grupo:12s} {sum(1 for g in GOLDEN if g['grupo'] == grupo)}")

# Las anclas se COMPRUEBAN: un folio del golden set que no exista en el corpus es un
# golden set roto, y hasta ahora nadie lo habría notado.
folios_corpus = set(FOLIOS)
huerfanos = {f for g in GOLDEN for f in g["rel"] if f not in folios_corpus}
assert not huerfanos, f"folios anotados que no están en el corpus: {sorted(huerfanos)}"
print(f"\ntodos los folios anotados existen en el corpus ✓")

14 preguntas · 12 respondibles · 2 negativas
   léxica       6
   vocabulario  6
   negativa     2

todos los folios anotados existen en el corpus ✓


## Parte 3 — Motor 1: BM25 desde cero

BM25 es el estándar léxico desde hace treinta años y sigue siendo el que hay que batir.
Puntúa un documento $d$ frente a una consulta $q$ con

$$\text{BM25}(q,d)=\sum_{t \in q} \text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}$$

Dos ideas, y las dos son correcciones a TF-IDF:

- **$k_1$ satura la frecuencia.** Que un término aparezca 20 veces no vale 20 veces más
  que una: a partir de cierto punto, deja de sumar.
- **$b$ normaliza por longitud.** Sin eso, los chunks largos ganan por tener más
  palabras. Con $b=0{,}75$ se penaliza a los que están por encima de la longitud media.

### ✏️ Ejercicio 3.1
Implementa la clase `BM25` con `puntuar(consulta)` que devuelva un score por chunk,
usando $k_1=1{,}5$ y $b=0{,}75$. El IDF con suavizado:
$\log\!\left(\frac{N-n_t+0{,}5}{n_t+0{,}5}+1\right)$ — el `+1` lo mantiene positivo.
Necesitas también `tokenizar(texto)` y, al final, el índice construido en `bm25`:
las celdas siguientes lo usan.

In [5]:
# Ejercicio 3.1 — BM25 Okapi en ~25 líneas
def tokenizar(t: str):
    return re.findall(r"[a-záéíóúñ0-9]+", t.lower())


class BM25:
    """Okapi BM25. k1 satura la frecuencia; b normaliza por longitud del documento."""

    def __init__(self, textos, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        docs = [tokenizar(t) for t in textos]
        self.N = len(docs)
        self.largos = [len(d) for d in docs]
        self.avg = sum(self.largos) / self.N
        self.tf = [Counter(d) for d in docs]
        df = Counter()
        for d in docs:
            df.update(set(d))          # DOCUMENT frequency: una vez por documento
        self.idf = {t: math.log((self.N - n + 0.5) / (n + 0.5) + 1) for t, n in df.items()}

    def puntuar(self, consulta):
        puntajes = np.zeros(self.N)
        for termino in tokenizar(consulta):
            idf = self.idf.get(termino)
            if idf is None:            # término que no está en el corpus: no aporta
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(termino, 0)
                if f:
                    denom = f + self.k1 * (1 - self.b + self.b * self.largos[i] / self.avg)
                    puntajes[i] += idf * f * (self.k1 + 1) / denom
        return puntajes


t0 = time.perf_counter()
bm25 = BM25(TEXTOS)
print(f"BM25 indexado en {time.perf_counter() - t0:.2f} s · {len(bm25.idf)} términos")

BM25 indexado en 0.03 s · 6679 términos


In [6]:
# Qué recupera, antes de creerse ningún promedio.
def top_bm25(consulta, n=3):
    p = bm25.puntuar(consulta)
    return sorted(range(len(p)), key=lambda i: -p[i])[:n], p


for consulta in ("What is byte pair encoding?",
                 "How can I make a model use less memory by lowering the precision of its weights?"):
    idxs, p = top_bm25(consulta)
    print(f"\n▸ {consulta}")
    for i in idxs:
        print(f"   p.{FOLIOS[i]:>3}  {p[i]:5.2f}  {TEXTOS[i][:80]}…")


▸ What is byte pair encoding?
   p.394  15.18  | Bidirectional Encoder Representations from | generating embeddings, 123 | |---…
   p.394  13.71  and, 88 | | Q-Former, 273-277 | prompt engineering, 178 | | BM25 algorithm, 233 …
   p. 51  13.17  a specific token for every sequence of whitespaces up to a list of 83 whitespace…

▸ How can I make a model use less memory by lowering the precision of its weights?
   p.364  21.29  ings in speed, storage, and compute as explained further in the previously refer…
   p.172  17.58  As shown in Figure 6-5, by lowering the value, it will consider fewer tokens and…
   p.367  14.74  Combined with the blockwise quantization, this normalization procedure allows fo…


## Parte 4 — Motor 2: búsqueda densa

El mismo corpus en un espacio de significado. Lo que BM25 no puede hacer por construcción:
emparejar «lowering the precision of its weights» con una página que dice *quantization*
sin compartir una sola palabra de contenido.

El modelo corre **en local** —954 chunks en unos 2 s—, así que no hace falta ni red ni
clave. Para usar el `bge-m3` de la H200 en su lugar (multilingüe, 1024 dimensiones), está
`curso/actividades/semana-2/h200-usfq-profesor.ipynb`: se cambia `codificar` y nada más.

In [7]:
try:
    from transformers.utils import logging as _hfl
    _hfl.disable_progress_bar()          # la barra de 199 tensores no dice nada útil
    import logging as _logging
    _logging.getLogger("huggingface_hub").setLevel(_logging.ERROR)
    from sentence_transformers import SentenceTransformer

    _modelo = SentenceTransformer("all-MiniLM-L6-v2")

    def codificar(textos):
        return np.asarray(_modelo.encode(textos, normalize_embeddings=True, batch_size=64))

    MOTOR_DENSO = "all-MiniLM-L6-v2"
except ImportError:
    # Respaldo sin dependencias: TF-IDF no es búsqueda densa y hay que decirlo en voz alta.
    from sklearn.feature_extraction.text import TfidfVectorizer

    _tfidf = TfidfVectorizer().fit(TEXTOS)

    def codificar(textos):
        M = _tfidf.transform(textos).toarray()
        return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

    MOTOR_DENSO = "TF-IDF (respaldo: NO es denso, comparte vocabulario con BM25)"

t0 = time.perf_counter()
V = codificar(TEXTOS)
print(f"motor denso: {MOTOR_DENSO}")
print(f"{V.shape[0]} chunks codificados en {time.perf_counter() - t0:.1f} s · "
      f"{V.shape[1]} dimensiones · normas a 1 → el producto punto ES el coseno")


def top_denso(consulta, n=3):
    q = codificar([consulta])[0]
    p = V @ q
    return list(np.argsort(-p)[:n]), p


for consulta in ("How can I make a model use less memory by lowering the precision of its weights?",
                 "Why do language models confidently state information that is false?"):
    idxs, p = top_denso(consulta)
    print(f"\n▸ {consulta}")
    for i in idxs:
        print(f"   p.{FOLIOS[i]:>3}  {p[i]:5.3f}  {TEXTOS[i][:80]}…")

motor denso: all-MiniLM-L6-v2
954 chunks codificados en 1.7 s · 384 dimensiones · normas a 1 → el producto punto ES el coseno

▸ How can I make a model use less memory by lowering the precision of its weights?
   p.370  0.492  This quantization procedure allows us to decrease the size of the original model…
   p.319  0.474  ``` from sentence_transformers import losses # Use the denoising auto-encoder lo…
   p.367  0.465  Combined with the blockwise quantization, this normalization procedure allows fo…

▸ Why do language models confidently state information that is false?
   p.296  0.510  model, you will often see data being used from natural language inference (NLI) …
   p.374  0.492  probability. In other words, the models should not be 'perplexed' when presented…
   p. 57  0.489  finding the best numerical representation for these tokens that the model can us…


## Parte 5 — Motor 3: híbrida con Reciprocal Rank Fusion

Los dos rankings anteriores no son comparables: BM25 puntúa en una escala sin techo y el
coseno va de −1 a 1. Normalizar para sumarlos es delicado y depende del corpus. RRF
esquiva el problema **tirando los scores y quedándose con las posiciones**:

$$\text{RRF}(d)=\sum_{r} \frac{1}{k_{\text{const}} + \text{pos}_r(d)}$$

con $k_{\text{const}}=60$, el valor del paper original. Lo que consigue: un documento que
sale 2.º en los dos rankings le gana a uno que sale 1.º en uno solo.

### ✏️ Ejercicio 5.1
Implementa `rrf(rankings, k_const=60)`, que recibe una lista de rankings (cada uno, una
lista de índices de chunk) y devuelve los índices fusionados. Cuidado con el desempate.
Escribe además `top_hibrido(consulta, n=3, candidatos=30)`, que fusiona los `candidatos`
primeros de BM25 y de la densa: las Partes 6 y 7 la usan.

In [8]:
# Ejercicio 5.1 — fusión por rango recíproco
def rrf(rankings, k_const=60):
    puntajes = {}
    for ranking in rankings:
        for posicion, idx in enumerate(ranking, start=1):
            puntajes[idx] = puntajes.get(idx, 0.0) + 1.0 / (k_const + posicion)
    # El desempate va por índice, y no es cosmético: con puntajes empatados el orden lo
    # decidiría el recorrido del diccionario, y dos corridas darían métricas distintas.
    return sorted(puntajes, key=lambda i: (-puntajes[i], i))


def top_hibrido(consulta, n=3, candidatos=30):
    idx_bm25, _ = top_bm25(consulta, candidatos)
    idx_denso, _ = top_denso(consulta, candidatos)
    return rrf([idx_bm25, idx_denso])[:n]


consulta = "How do I measure the quality of a search system?"
print(f"▸ {consulta}\n")
print(f"   {'BM25':>18} {'densa':>18} {'híbrida (RRF)':>18}")
b, _ = top_bm25(consulta, 3)
d, _ = top_denso(consulta, 3)
h = top_hibrido(consulta, 3)
for i in range(3):
    print(f"   {'p.' + str(FOLIOS[b[i]]):>18} {'p.' + str(FOLIOS[d[i]]):>18} "
          f"{'p.' + str(FOLIOS[h[i]]):>18}")

▸ How do I measure the quality of a search system?

                 BM25              densa      híbrida (RRF)
                p.246              p.244              p.247
                p.255              p.245              p.246
                p.250              p.249              p.248


## Parte 6 — Motor 4: reranking con un cross-encoder

Los tres motores anteriores comparan **representaciones calculadas por separado**: el
chunk se codificó sin saber cuál sería la pregunta. Un *cross-encoder* hace lo contrario
—lee pregunta y chunk **juntos**, en la misma pasada— y por eso puntúa mucho mejor… y por
eso no se puede usar para buscar: habría que evaluarlo contra los 954 chunks en cada
consulta.

El patrón es siempre el mismo y se llama *retrieve and rerank*: un motor barato trae 30
candidatos, y el caro reordena esos 30. Es el capítulo 8 del libro que estamos indexando.

In [9]:
try:
    from sentence_transformers import CrossEncoder
    _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    HAY_RERANKER = True
    print("reranker: cross-encoder/ms-marco-MiniLM-L-6-v2")
except Exception as e:
    HAY_RERANKER = False
    print(f"sin reranker ({type(e).__name__}): la Parte 7 compara solo los tres primeros")


def top_rerank(consulta, n=3, candidatos=30):
    """Trae `candidatos` con el híbrido y los reordena leyéndolos junto a la pregunta."""
    cand = top_hibrido(consulta, candidatos, candidatos)
    if not HAY_RERANKER:
        return cand[:n]
    puntajes = _reranker.predict([(consulta, TEXTOS[i]) for i in cand])
    return [cand[i] for i in np.argsort(-puntajes)][:n]


if HAY_RERANKER:
    consulta = "What is byte pair encoding?"
    t0 = time.perf_counter()
    antes = top_hibrido(consulta, 3)
    despues = top_rerank(consulta, 3)
    print(f"\n▸ {consulta}   (reordenar 30 candidatos: {time.perf_counter() - t0:.2f} s)")
    print(f"   híbrida  : {[f'p.{FOLIOS[i]}' for i in antes]}")
    print(f"   +rerank  : {[f'p.{FOLIOS[i]}' for i in despues]}")
    print(f"   el libro explica BPE en los folios 43-55; el 394 es su índice alfabético")

reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

▸ What is byte pair encoding?   (reordenar 30 candidatos: 0.14 s)
   híbrida  : ['p.394', 'p.49', 'p.55']
   +rerank  : ['p.53', 'p.55', 'p.394']
   el libro explica BPE en los folios 43-55; el 394 es su índice alfabético


## Parte 7 — Los cuatro motores, con números

### ✏️ Ejercicio 7.1
Evalúa los cuatro con `hit_rate_k`, `recall_k` y `mrr` sobre `RESPONDIBLES`, con k=5.
El ranking que las métricas esperan es de **folios deduplicados** (varios chunks de la
misma página cuentan una vez), no de índices de chunk. Mide también el tiempo por
consulta y la abstención de cada motor sobre las dos negativas.

In [10]:
# Ejercicio 7.1 — la tabla comparativa
def ranking_folios(idxs):
    """De índices de chunk a folios deduplicados, conservando el orden."""
    vistos, orden = set(), []
    for i in idxs:
        if FOLIOS[i] not in vistos:
            vistos.add(FOLIOS[i])
            orden.append(FOLIOS[i])
    return orden


MOTORES = {
    "BM25": lambda q: top_bm25(q, 30)[0],
    "densa": lambda q: top_denso(q, 30)[0],
    "híbrida (RRF)": lambda q: top_hibrido(q, 30, 30),
    "+ reranking": lambda q: top_rerank(q, 30, 30),
}
if not HAY_RERANKER:
    MOTORES.pop("+ reranking")

tabla = {}
for nombre, motor in MOTORES.items():
    t0 = time.perf_counter()
    rankings = [ranking_folios(motor(g["q"])) for g in GOLDEN]
    segundos = (time.perf_counter() - t0) / len(GOLDEN)
    relevantes = [g["rel"] for g in GOLDEN]
    tabla[nombre] = {
        f"hit@{K}": hit_rate_k(rankings, relevantes, K),
        f"recall@{K}": recall_k(rankings, relevantes, K),
        "mrr": mrr(rankings, relevantes, K),
        "s/consulta": segundos,
        "rankings": rankings,
    }

print(f"{'motor':>15} {'hit@5':>7} {'recall@5':>9} {'mrr':>7} {'s/consulta':>11}")
for nombre, r in tabla.items():
    print(f"{nombre:>15} {r[f'hit@{K}']:>7.2f} {r[f'recall@{K}']:>9.3f} {r['mrr']:>7.3f} "
          f"{r['s/consulta']:>11.3f}")

          motor   hit@5  recall@5     mrr  s/consulta
           BM25    0.83     0.485   0.531       0.002
          densa    0.92     0.528   0.732       0.007
  híbrida (RRF)    1.00     0.704   0.854       0.009
    + reranking    0.92     0.641   0.799       0.083


In [11]:
# Y dónde gana cada uno, que es lo que el promedio esconde.
print(f"{'pregunta':56s} " + " ".join(f"{n:>14}" for n in tabla))
for j, g in enumerate(GOLDEN):
    if not g["rel"]:
        continue
    celdas = []
    for nombre, r in tabla.items():
        pos = next((p for p, f in enumerate(r["rankings"][j][:K], 1) if f in g["rel"]), None)
        celdas.append(f"{('pos ' + str(pos)) if pos else '—':>14}")
    print(f"{g['q'][:54]:56s} " + " ".join(celdas))
print(f"\n«pos n» = el primer folio correcto apareció en el puesto n de los {K} primeros")

pregunta                                                           BM25          densa  híbrida (RRF)    + reranking
What is byte pair encoding?                                       pos 2          pos 5          pos 2          pos 1
What is negative sampling in word2vec training?                   pos 1          pos 1          pos 1          pos 1
What is the difference between a bi-encoder and a cros            pos 1          pos 1          pos 1          pos 1
What is dense retrieval?                                              —          pos 4          pos 4          pos 3
What is reranking and when should I use it?                           —          pos 1          pos 1          pos 1
What is a vector database used for?                               pos 2          pos 1          pos 1          pos 1
How can I make a model use less memory by lowering the            pos 3          pos 3          pos 1          pos 4
How do I adapt a large model without updating all of i          

### Lo que dicen los números, incluido lo que incomoda

En la medición de referencia (2026-09-23) el orden fue **BM25 < densa < híbrida**, que es
lo esperable y lo que justifica el híbrido del Taller 2. Pero **el reranking no mejoró al
híbrido**: lo empató o lo bajó.

Eso no es un fallo del cuaderno ni un error de implementación, y no se tapa. Un
cross-encoder entrenado sobre **MS MARCO** vio consultas cortas de buscador web contra
pasajes web; aquí recibe preguntas largas contra prosa de un libro técnico. La pregunta
que hay que llevarse no es «¿el reranking sirve?» sino **«¿sobre qué distribución se
entrenó este reranker, y se parece a la mía?»** — y la forma de contestarla es esta tabla,
no la documentación del modelo.

Donde el reranking **sí** cambia las cosas es en las preguntas léxicas contaminadas por el
índice alfabético del libro: BM25 pone arriba el folio 394 —que solo lista la palabra— y
el reranker lo hunde porque, leído junto a la pregunta, no responde nada.

## Parte 8 — Las negativas: saber callarse

### ✏️ Ejercicio 8.1
Las dos preguntas negativas no están en el libro. Con el score top-1 del motor denso
(coseno, comparable entre consultas), elige un umbral mirando los valores y calcula las
dos tasas de `tasa_abstencion`. ¿Hay algún umbral que separe las negativas de las
respondibles sin sacrificar ninguna?

In [12]:
# Ejercicio 8.1 — el umbral de abstención sobre el corpus real
scores_top1 = []
for g in GOLDEN:
    _, p = top_denso(g["q"], 1)
    scores_top1.append(float(np.max(p)))

print(f"{'grupo':12s} {'score top-1':>12s}   pregunta")
for g, s in sorted(zip(GOLDEN, scores_top1), key=lambda x: -x[1]):
    print(f"{g['grupo']:12s} {s:>12.3f}   {g['q'][:58]}")

peor_respondible = min(s for g, s in zip(GOLDEN, scores_top1) if g["rel"])
mejor_negativa = max(s for g, s in zip(GOLDEN, scores_top1) if not g["rel"])
print(f"\npeor respondible : {peor_respondible:.3f}")
print(f"mejor negativa   : {mejor_negativa:.3f}")
if mejor_negativa < peor_respondible:
    UMBRAL = (mejor_negativa + peor_respondible) / 2
    print(f"→ SÍ hay umbral limpio: {UMBRAL:.3f}")
else:
    UMBRAL = peor_respondible
    print(f"→ NO hay umbral limpio: se solapan. Cualquier corte sacrifica algo, y elegirlo "
          f"es una decisión de producto, no de código.")
print("  " + str({k: round(v, 2) for k, v in
                  tasa_abstencion(scores_top1, [g["rel"] for g in GOLDEN], UMBRAL).items()}))

grupo         score top-1   pregunta
léxica              0.696   What is negative sampling in word2vec training?
vocabulario         0.655   How do I measure the quality of a search system?
léxica              0.615   What is reranking and when should I use it?
léxica              0.583   What is dense retrieval?
vocabulario         0.563   What happens when keyword search misses a relevant documen
léxica              0.547   What is the difference between a bi-encoder and a cross-en
vocabulario         0.510   Why do language models confidently state information that 
vocabulario         0.506   How should I split long documents before indexing them?
léxica              0.503   What is a vector database used for?
vocabulario         0.499   How do I adapt a large model without updating all of its p
vocabulario         0.492   How can I make a model use less memory by lowering the pre
léxica              0.398   What is byte pair encoding?
negativa            0.281   How much does Qdra

## Parte 9 — Cerrar el pipeline: responder citando el folio

Recuperar no era el objetivo: era la mitad cara de acertar. Esta parte engancha el mejor
motor de la tabla con el generador y **obliga a citar el folio**, que es lo único que
convierte una respuesta en algo verificable: quien la lea puede abrir el PDF por esa
página y comprobarla.

La conexión es la misma del miércoles —OpenAI por HTTP, sin SDK, con la clave del `.env`
de la raíz— y con los mismos dos cuidados: `max_completion_tokens` (esta familia rechaza
`max_tokens`) y sin `temperature` (solo admite su valor por defecto).

In [13]:
import json
import os
import urllib.error
import urllib.request

MODELO_API = "gpt-5.6-terra"
MAX_SALIDA = 800

_env = RAIZ / ".env"
if _env.is_file():
    for _linea in _env.read_text(encoding="utf-8").splitlines():
        _linea = _linea.strip()
        if _linea and not _linea.startswith("#") and "=" in _linea:
            _k, _, _v = _linea.removeprefix("export ").partition("=")
            _k, _v = _k.strip(), _v.strip().strip('"').strip("'")
            if _k and _v:
                os.environ.setdefault(_k, _v)
print(f"OPENAI_API_KEY : {'presente' if os.environ.get('OPENAI_API_KEY') else 'ausente — esta parte se salta'}")


def responder(pregunta, k=4):
    """Recupera con el mejor motor y responde citando el folio de cada afirmación."""
    idxs = top_rerank(pregunta, k, 30) if HAY_RERANKER else top_hibrido(pregunta, k, 30)
    contexto = "\n\n".join(f"[p.{FOLIOS[i]}] {TEXTOS[i]}" for i in idxs)
    prompt = (f"Fragmentos de un libro técnico:\n\n{contexto}\n\n"
              f"Pregunta: {pregunta}\n\n"
              f"Responde en español y en dos frases, SOLO con lo que digan los fragmentos, "
              f"citando el folio entre corchetes como [p.123]. Si no alcanzan, dilo.")
    if not os.environ.get("OPENAI_API_KEY"):
        print(f"   (sin clave: se muestra qué se recuperó) folios {[FOLIOS[i] for i in idxs]}")
        return None
    cuerpo = json.dumps({"model": MODELO_API,
                         "messages": [{"role": "user", "content": prompt}],
                         "max_completion_tokens": MAX_SALIDA}).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=cuerpo,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
    try:
        with urllib.request.urlopen(req, timeout=90) as r:
            d = json.loads(r.read())
    except urllib.error.HTTPError as e:
        print(f"   [API] {e.code} · {e.read().decode()[:160]}")
        return None
    uso = d.get("usage", {})
    print(f"   [API] {MODELO_API} · {uso.get('prompt_tokens')} tok entrada / "
          f"{uso.get('completion_tokens')} salida · folios {[FOLIOS[i] for i in idxs]}")
    return d["choices"][0]["message"]["content"].strip()


for pregunta in ("What is the difference between a bi-encoder and a cross-encoder?",
                 "How much does Qdrant Cloud cost per month?"):      # la segunda no está
    print(f"\n▸ {pregunta}")
    texto = responder(pregunta)
    if texto:
        print(f"   {texto}")

OPENAI_API_KEY : presente

▸ What is the difference between a bi-encoder and a cross-encoder?
   [API] 401 · {
  "error": {
    "message": "Your API key has been invalidated.",
    "type": null,
    "code": "token_invalidated",
    "param": null
  },
  "status": 401
}

▸ How much does Qdrant Cloud cost per month?
   [API] 401 · {
  "error": {
    "message": "Your API key has been invalidated.",
    "type": null,
    "code": "token_invalidated",
    "param": null
  },
  "status": 401
}


## Cierre — checklist para el sábado (Taller 2)

- [ ] Corpus real elegido y **troceado conservando la referencia** (folio, sección o URL):
      sin eso no hay cita, y sin cita no hay RAG que defender
- [ ] Golden set con los tres grupos: léxicas, de vocabulario distinto y negativas
- [ ] Las anclas del golden set **comprobadas contra el corpus** (el `assert` de la Parte 2)
- [ ] Los cuatro motores corriendo, con su tabla de Hit Rate, Recall@k, MRR y tiempo
- [ ] Docker con Qdrant probado (`docker run -p 6333:6333 qdrant/qdrant`)
- [ ] Extensión elegida CON evidencia de esta tabla, no por reputación del método